# Avant le TP6 : lire une métrique, de l'agent JMX jusqu'à l'alerte

Le TP6 interroge la chaîne de supervision du lab depuis le notebook. Cette démonstration suit une seule métrique de la JVM, le nombre de threads, à chaque maillon :

1. telle que l'agent JMX d'un nœud l'expose, en texte ;
2. telle que Prometheus la renvoie à une requête PromQL, en JSON ;
3. puis les règles d'alerte et leur état, telles que Prometheus les publie.

Chaque étape montre la version **terminal**, puis la version **Python**. Ici, tout est en lecture : les deux versions sont exécutées. En terminal, les pages HTTP se lisent avec `wget`, lancé dans le conteneur `prometheus` (les conteneurs `jupyter` et `tools` n'ont ni `curl` ni `wget`).

Sorties de référence : exécution du 12/11/2026 22:33 UTC sur le lab (mode `extended`).

Les adresses sont définies une seule fois, ici. Depuis le conteneur `prometheus`, Prometheus lui-même répond sur `localhost:9090`. Dans un terminal de JupyterLab, définissez d'abord : `export AGENT=http://kafka-2:9404/metrics METRIC=jvm_threads_current PROMETHEUS_LOCAL=http://localhost:9090`.

In [1]:
import requests

PROMETHEUS = "http://prometheus:9090"
PROMETHEUS_LOCAL = "http://localhost:9090"  # le même, vu depuis le conteneur prometheus
AGENT = "http://kafka-2:9404/metrics"
METRIC = "jvm_threads_current"
DELAY = 10  # secondes


def get_json(path, **params):
    """Interroge l'API de Prometheus ; lève une erreur si la réponse n'est pas un succès."""
    response = requests.get(PROMETHEUS + path, params=params, timeout=DELAY)
    response.raise_for_status()
    body = response.json()
    if body["status"] != "success":
        raise RuntimeError(body.get("error", body))
    return body["data"]

## 1. L'agent JMX : du texte

Chaque nœud publie ses métriques sur le port 9404. Une métrique, c'est une ligne `HELP` (description), une ligne `TYPE` (`gauge` pour une valeur instantanée, `counter` pour un total qui ne fait que croître), puis une ligne par série : `nom{etiquettes} valeur`.

**En terminal**

In [2]:
!docker exec prometheus wget -qO- $AGENT | grep $METRIC

# HELP jvm_threads_current Current thread count of a JVM
# TYPE jvm_threads_current gauge
jvm_threads_current 112.0


**En Python**

In [3]:
response = requests.get(AGENT, timeout=DELAY)
response.raise_for_status()
text = response.text
for line in text.splitlines():
    if METRIC in line:
        print(line)
print(len(text.splitlines()), "lignes au total")

# HELP jvm_threads_current Current thread count of a JVM
# TYPE jvm_threads_current gauge
jvm_threads_current 112.0
7096 lignes au total


## 2. Prometheus : du JSON

Prometheus collecte ces pages toutes les 15 s et ajoute les étiquettes `job` et `instance`. Son API répond aux requêtes PromQL. La valeur arrive sous forme de chaîne, avec son horodatage.

**En terminal** : la réponse brute, en JSON, sur une ligne.

In [4]:
!docker exec prometheus wget -qO- "$PROMETHEUS_LOCAL/api/v1/query?query=$METRIC"

{"status":"success","data":{"resultType":"vector","result":[{"metric":{"__name__":"jvm_threads_current","instance":"kafka-3","job":"kafka"},"value":[1794523413.763,"111"]},{"metric":{"__name__":"jvm_threads_current","instance":"kafka-2","job":"kafka"},"value":[1794523413.763,"112"]},{"metric":{"__name__":"jvm_threads_current","instance":"loadgen-consumer","job":"kafka-clients"},"value":[1794523413.763,"13"]},{"metric":{"__name__":"jvm_threads_current","instance":"kafka-1","job":"kafka"},"value":[1794523413.763,"114"]},{"metric":{"__name__":"jvm_threads_current","instance":"loadgen-producer","job":"kafka-clients"},"value":[1794523413.763,"13"]}]}}

**En Python** : `get_json()` vérifie le statut de la réponse, et on parcourt le résultat.

In [5]:
data = get_json("/api/v1/query", query=METRIC)
print(data["resultType"])
for result in data["result"]:
    print(result["metric"]["job"], result["metric"]["instance"], result["value"])

vector
kafka kafka-3 [1794523413.878, '111']
kafka kafka-2 [1794523413.878, '112']
kafka-clients loadgen-consumer [1794523413.878, '13']
kafka kafka-1 [1794523413.878, '114']
kafka-clients loadgen-producer [1794523413.878, '13']


Une requête peut agréger : ici, le total des threads par job. En terminal, les espaces et les parenthèses de la requête sont encodés dans l'URL (`%20`, `%28`, `%29`) ; en Python, `requests` le fait seul.

**En terminal**

In [6]:
!docker exec prometheus wget -qO- "$PROMETHEUS_LOCAL/api/v1/query?query=sum%20by%20%28job%29%20%28$METRIC%29"

{"status":"success","data":{"resultType":"vector","result":[{"metric":{"job":"kafka"},"value":[1794523413.923,"337"]},{"metric":{"job":"kafka-clients"},"value":[1794523413.923,"26"]}]}}

**En Python**

In [7]:
for result in get_json("/api/v1/query", query=f"sum by (job) ({METRIC})")["result"]:
    print(result["metric"]["job"], float(result["value"][1]))

kafka 337.0
kafka-clients 26.0


## 3. Les règles d'alerte

Prometheus évalue ses règles toutes les 15 s. Chaque règle a un état : `inactive`, `pending` (condition vraie, délai `for` pas encore écoulé) ou `firing`.

**En terminal** : la réponse complète est longue ; on n'en garde que les noms et les états.

In [8]:
!docker exec prometheus wget -qO- "$PROMETHEUS_LOCAL/api/v1/rules" | grep -oE '"state":"[a-z]+","name":"[A-Za-z]+"'

"state":"inactive","name":"KafkaNodeDown"
"state":"inactive","name":"KafkaNoActiveController"
"state":"inactive","name":"KafkaOfflinePartitions"
"state":"inactive","name":"KafkaUnderMinIsrPartitions"
"state":"inactive","name":"KafkaUnderReplicatedPartitions"
"state":"inactive","name":"KafkaFencedBrokers"
"state":"inactive","name":"KafkaRequestHandlerSaturated"
"state":"inactive","name":"KafkaConsumerGroupLagHigh"
"state":"inactive","name":"KafkaBrokerLogSizeHigh"


**En Python**

In [9]:
for group in get_json("/api/v1/rules")["groups"]:
    for rule in group["rules"]:
        print("%-25s %-40s %s" % (group["name"], rule["name"], rule["state"]))

kafka-cluster             KafkaNodeDown                            inactive
kafka-cluster             KafkaNoActiveController                  inactive
kafka-cluster             KafkaOfflinePartitions                   inactive
kafka-cluster             KafkaUnderMinIsrPartitions               inactive
kafka-cluster             KafkaUnderReplicatedPartitions           inactive
kafka-cluster             KafkaFencedBrokers                       inactive
kafka-cluster             KafkaRequestHandlerSaturated             inactive
kafka-consumers           KafkaConsumerGroupLagHigh                inactive
kafka-storage             KafkaBrokerLogSizeHigh                   inactive


## La méthode du TP6

Ouvrez `tps/jour2/tp6_observabilite/enonce.ipynb`.

- Vous écrirez les fonctions qui lisent chaque maillon : `metrics()` pour l'agent, `promql()` pour Prometheus, `alert_states()` pour les alertes. Elles servent ensuite jusqu'à la fin du TP.
- Gardez Grafana ouvert dans le navigateur (http://localhost:3000) : chaque valeur calculée dans le notebook a sa tuile.
- Deux cellules attendent une alerte (6.7 et 6.8) : 2 à 4 minutes en exécution, à ne pas interrompre.
- Le lab doit être en mode `extended`.